# Lab 8 — SOLUTION

In [ ]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import recall_score, accuracy_score
from fairlearn.metrics import MetricFrame

df = pd.read_csv("data/adult_like.csv")
X = pd.get_dummies(df.drop(columns=["income_gt_50k"]), drop_first=True)
y, g = df["income_gt_50k"], df["group"]
Xtr, Xte, ytr, yte, gtr, gte = train_test_split(X, y, g, test_size=0.3, random_state=42)

def tpr_gap(pred):
    mf = MetricFrame(metrics=recall_score, y_true=yte, y_pred=pred, sensitive_features=gte)
    return mf.difference()

base = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr)
print("baseline TPR gap:", round(tpr_gap(base.predict(Xte)), 3))

# reweighing: weight = P(group)P(outcome) / P(group,outcome)
n = len(ytr)
w = np.ones(n)
for grp in ["A", "B"]:
    for out in [0, 1]:
        mask = (gtr == grp) & (ytr == out)
        p_g = (gtr == grp).mean(); p_o = (ytr == out).mean(); p_go = mask.mean()
        w[mask.values] = p_g * p_o / p_go
mit = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr, logisticregression__sample_weight=w)
print("reweighed TPR gap:", round(tpr_gap(mit.predict(Xte)), 3))
print("accuracy before/after:", round(accuracy_score(yte, base.predict(Xte)),3), round(accuracy_score(yte, mit.predict(Xte)),3))

**Teaching notes:** the trade-off sentence is the graded part: 'TPR gap narrowed by X points at a cost of Y points of accuracy — a documented values decision.'